# Gujarat Hourly LSTM Training (Residual Forecast)

This notebook trains an LSTM on `gujarat_hourly_merged.csv` using the raw demand history plus weather and calendar features.

The target is the next-hour demand residual, so the model learns deviations from persistence instead of only smoothing the series.

In [ ]:
import copy
import math
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATA_CANDIDATES = [
    Path('guarat_hourly_demand.csv'),
    Path('gujarat_hourly_merged.csv'),
    Path.cwd() / 'guarat_hourly_demand.csv',
    Path.cwd() / 'gujarat_hourly_merged.csv',
    Path(r'd:\\project course\\guarat_hourly_demand.csv'),
    Path(r'd:\\project course\\gujarat_hourly_merged.csv'),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not find guarat_hourly_demand.csv or gujarat_hourly_merged.csv in expected paths.')

ARTIFACT_DIR = Path('models')
SCALER_DIR = Path('scalars')
ARTIFACT_DIR.mkdir(exist_ok=True)
SCALER_DIR.mkdir(exist_ok=True)

BASELINE_START = pd.Timestamp('2020-01-01 00:00:00')
BASELINE_END = pd.Timestamp('2024-06-30 23:59:59')
WALK_FORWARD_START = pd.Timestamp('2024-07-01 00:00:00')
WALK_FORWARD_END = pd.Timestamp('2025-06-30 23:59:59')

LOOKBACK = 168
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
BATCH_SIZE = 256
EPOCHS = 100
PATIENCE = 8
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP = 1.0
OVERFITTING_THRESHOLD = 1.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

In [ ]:
df_all = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
df_all = df_all.sort_values('datetime').reset_index(drop=True)

# Backward-compatible fallbacks if the expanded weather columns are not yet present in the dataset.
if 'weather_relative_humidity_2m' not in df_all.columns:
    temp = df_all['weather_temperature_2m']
    dewpoint = df_all['weather_dewpoint_2m']
    saturation_vp = 6.112 * np.exp((17.67 * temp) / (temp + 243.5))
    actual_vp = 6.112 * np.exp((17.67 * dewpoint) / (dewpoint + 243.5))
    df_all['weather_relative_humidity_2m'] = np.clip((actual_vp / saturation_vp) * 100.0, 0.0, 100.0)
if 'weather_precipitation' not in df_all.columns:
    df_all['weather_precipitation'] = 0.0

df_all['thermal_stress'] = df_all['weather_temperature_2m'] - df_all['weather_apparent_temperature']
df_all['humidity_temp_index'] = df_all['weather_temperature_2m'] * df_all['weather_relative_humidity_2m']
df_all['precipitation_flag'] = (df_all['weather_precipitation'] > 0).astype(float)
df_all['precipitation_x_cloud'] = df_all['weather_precipitation'] * df_all['weather_cloudcover']
df_all['weather_energy_interaction'] = df_all['weather_temperature_2m'] * (1.0 + df_all['weather_shortwave_radiation'] / 1000.0)
df_all['weather_motion_interaction'] = df_all['weather_shortwave_radiation'] * df_all['weather_windspeed_10m']
df_all['humidity_stress'] = df_all['weather_relative_humidity_2m'] * (df_all['weather_temperature_2m'] - df_all['weather_dewpoint_2m'])

df_all['target_demand_mw'] = df_all['demand_mw'].shift(-1)
df_all['target_delta_mw'] = df_all['target_demand_mw'] - df_all['demand_mw']
df_all = df_all.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

feature_columns = [
    'demand_mw',
    'weather_temperature_2m',
    'weather_relative_humidity_2m',
    'weather_shortwave_radiation',
    'weather_precipitation',
    'weather_windspeed_10m',
    'weather_dewpoint_2m',
    'weather_cloudcover',
    'weather_apparent_temperature',
    'hour',
    'day_of_week',
    'day_of_year',
    'month',
    'is_weekend',
    'is_sunday',
    'year_index',
    'sin_hour',
    'cos_hour',
    'sin_month',
    'cos_month',
    'sin_day_year',
    'cos_day_year',
    'is_peak_season',
    'thermal_stress',
    'humidity_temp_index',
    'precipitation_flag',
    'precipitation_x_cloud',
    'weather_energy_interaction',
    'weather_motion_interaction',
    'humidity_stress',
]
feature_columns = [col for col in feature_columns if col in df_all.columns]
target_column = 'target_delta_mw'

df = df_all[(df_all['datetime'] >= BASELINE_START) & (df_all['datetime'] <= BASELINE_END)].copy().reset_index(drop=True)
walk_forward_df = df_all[(df_all['datetime'] >= WALK_FORWARD_START) & (df_all['datetime'] <= WALK_FORWARD_END)].copy().reset_index(drop=True)

if df.empty:
    raise ValueError('No rows found in baseline horizon 2020-01-01 to 2024-06-30.')

print('Loaded file:', DATA_PATH)
print('All cleaned rows:', len(df_all))
print('All cleaned range:', df_all['datetime'].min(), '->', df_all['datetime'].max())
print('Baseline rows:', len(df))
print('Baseline range:', df['datetime'].min(), '->', df['datetime'].max())
print('Walk-forward reserved rows:', len(walk_forward_df))
if len(walk_forward_df) > 0:
    print('Walk-forward reserved range:', walk_forward_df['datetime'].min(), '->', walk_forward_df['datetime'].max())
print('Feature count:', len(feature_columns))
print('Feature columns:', feature_columns)
print('Target column:', target_column)

In [ ]:
values = df[feature_columns].to_numpy(dtype=np.float32)
targets = df[target_column].to_numpy(dtype=np.float32).reshape(-1, 1)
current_demand = df['demand_mw'].to_numpy(dtype=np.float32)

n_rows = len(df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))

x_scaler = StandardScaler()
y_scaler = StandardScaler()
x_scaler.fit(values[:train_end])
y_scaler.fit(targets[:train_end])

x_scaled = x_scaler.transform(values).astype(np.float32)
y_scaled = y_scaler.transform(targets).astype(np.float32).reshape(-1)

def build_sequences(feature_array, target_array, timestamps, demand_array, lookback):
    sequences = []
    labels = []
    sequence_times = []
    baselines = []
    for end_idx in range(lookback - 1, len(feature_array)):
        start_idx = end_idx - lookback + 1
        sequences.append(feature_array[start_idx:end_idx + 1])
        labels.append(target_array[end_idx])
        sequence_times.append(timestamps[end_idx])
        baselines.append(demand_array[end_idx])
    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(labels, dtype=np.float32),
        np.asarray(sequence_times),
        np.asarray(baselines, dtype=np.float32),
    )

timestamps = df['datetime'].to_numpy()
X, y, sequence_times, baseline_demand = build_sequences(x_scaled, y_scaled, timestamps, current_demand, LOOKBACK)
target_positions = np.arange(LOOKBACK - 1, n_rows)

train_mask = target_positions < train_end
val_mask = (target_positions >= train_end) & (target_positions < val_end)
test_mask = target_positions >= val_end

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]
base_train = baseline_demand[train_mask]
base_val = baseline_demand[val_mask]
base_test = baseline_demand[test_mask]
time_train = sequence_times[train_mask]
time_val = sequence_times[val_mask]
time_test = sequence_times[test_mask]

print('Baseline train/val/test split boundaries:')
print('  Train end datetime:', df.iloc[train_end - 1]['datetime'])
print('  Validation end datetime:', df.iloc[val_end - 1]['datetime'])
print('  Baseline test end datetime:', df.iloc[-1]['datetime'])
print('Reserved walk-forward horizon:', WALK_FORWARD_START, 'to', WALK_FORWARD_END)

print('Sequence shapes:')
print('  X_train:', X_train.shape, 'y_train:', y_train.shape)
print('  X_val  :', X_val.shape, 'y_val  :', y_val.shape)
print('  X_test :', X_test.shape, 'y_test :', y_test.shape)

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
train_eval_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
class RegularizedLSTM(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, 96),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.Dropout(0.30)
        )
        self.lstm = nn.LSTM(
            input_size=96,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.35,
            bidirectional=False
        )
        self.bn_lstm = nn.BatchNorm1d(128)
        self.head = nn.Sequential(
            nn.Linear(128, 96),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(96, 48),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(48, 1)
        )

    def forward(self, x):
        x = self.input_proj(x.reshape(-1, x.shape[-1])).reshape(x.shape[0], x.shape[1], -1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.bn_lstm(x)
        return self.head(x).squeeze(-1)

model = RegularizedLSTM(input_dim=X_train.shape[-1]).to(DEVICE)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {param_count:,}')

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_items = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
        total_items += batch_x.size(0)
    return total_loss / max(total_items, 1)

best_state = None
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'overfitting_ratio': []}

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    overfitting_ratio = val_loss / (train_loss + 1e-6) if train_loss > 0 else 0
    history['overfitting_ratio'].append(overfitting_ratio)

    improved = val_loss < best_val_loss - 1e-4
    if improved:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    current_lr = optimizer.param_groups[0]['lr']
    status = 'BEST' if improved else ('OVERFIT' if overfitting_ratio > OVERFITTING_THRESHOLD else 'OK')
    print(f'Epoch {epoch:03d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | ratio={overfitting_ratio:.2f} | lr={current_lr:.2e} | {status}')

    if patience_counter >= PATIENCE:
        print('Early stopping triggered.')
        break

if best_state is not None:
    model.load_state_dict(best_state)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train loss', linewidth=2)
plt.plot(history['val_loss'], label='Validation loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('LSTM Training Curve')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history['overfitting_ratio'], label='Val / Train ratio', linewidth=2, color='red')
plt.axhline(y=OVERFITTING_THRESHOLD, color='orange', linestyle='--', label=f'Threshold ({OVERFITTING_THRESHOLD})')
plt.xlabel('Epoch')
plt.ylabel('Loss Ratio')
plt.title('Overfitting Indicator')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def predict(loader):
    model.eval()
    pred_deltas = []
    actual_deltas = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            outputs = model(batch_x).cpu().numpy()
            pred_deltas.append(outputs)
            actual_deltas.append(batch_y.numpy())
    pred_deltas = np.concatenate(pred_deltas)
    actual_deltas = np.concatenate(actual_deltas)
    return pred_deltas, actual_deltas

def to_actual_demand(delta_values, baseline_values):
    delta_values = y_scaler.inverse_transform(delta_values.reshape(-1, 1)).reshape(-1)
    return baseline_values + delta_values

def compute_metrics(actual_orig, pred_orig):
    rmse = math.sqrt(mean_squared_error(actual_orig, pred_orig))
    mae = mean_absolute_error(actual_orig, pred_orig)
    mape = np.mean(np.abs((actual_orig - pred_orig) / np.maximum(np.abs(actual_orig), 1e-6))) * 100
    r2 = r2_score(actual_orig, pred_orig)

    p90_threshold = np.percentile(actual_orig, 90)
    peak_mask = actual_orig >= p90_threshold
    peak_mape = (
        np.mean(
            np.abs(
                (actual_orig[peak_mask] - pred_orig[peak_mask]) /
                np.maximum(np.abs(actual_orig[peak_mask]), 1e-6)
            )
        ) * 100
    ) if peak_mask.sum() > 0 else 0

    return {
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'r2': r2,
        'peak_mape': peak_mape,
        'peak_samples': peak_mask.sum(),
    }

train_pred_delta, train_actual_delta = predict(train_eval_loader)
val_pred_delta, val_actual_delta = predict(val_loader)
test_pred_delta, test_actual_delta = predict(test_loader)

train_preds = to_actual_demand(train_pred_delta, base_train)
train_actuals = to_actual_demand(train_actual_delta, base_train)
val_preds = to_actual_demand(val_pred_delta, base_val)
val_actuals = to_actual_demand(val_actual_delta, base_val)
test_preds = to_actual_demand(test_pred_delta, base_test)
test_actuals = to_actual_demand(test_actual_delta, base_test)

train_metrics = compute_metrics(train_actuals, train_preds)
val_metrics = compute_metrics(val_actuals, val_preds)
test_metrics = compute_metrics(test_actuals, test_preds)

print('\n' + '='*60)
print('--- Train Set Metrics ---')
print(f'  RMSE: {train_metrics["rmse"]:.3f}')
print(f'  MSE: {train_metrics["rmse"]**2:.3f}')
print(f'  MAE : {train_metrics["mae"]:.3f}')
print(f'  MAPE: {train_metrics["mape"]:.3f}%')
print(f'  Peak MAPE (>90% Actuals): {train_metrics["peak_mape"]:.3f}%')
print(f'  R^2 : {train_metrics["r2"]:.4f}')
print(f'  Samples used for Peak MAPE: {train_metrics["peak_samples"]} / {len(train_actuals)}')

print('\n--- Validation Set Metrics ---')
print(f'  RMSE: {val_metrics["rmse"]:.3f}')
print(f'  MSE: {val_metrics["rmse"]**2:.3f}')
print(f'  MAE : {val_metrics["mae"]:.3f}')
print(f'  MAPE: {val_metrics["mape"]:.3f}%')
print(f'  Peak MAPE (>90% Actuals): {val_metrics["peak_mape"]:.3f}%')
print(f'  R^2 : {val_metrics["r2"]:.4f}')
print(f'  Samples used for Peak MAPE: {val_metrics["peak_samples"]} / {len(val_actuals)}')

print('\n--- Test Set Metrics ---')
print(f'  RMSE: {test_metrics["rmse"]:.3f}')
print(f'  MSE: {test_metrics["rmse"]**2:.3f}')
print(f'  MAE : {test_metrics["mae"]:.3f}')
print(f'  MAPE: {test_metrics["mape"]:.3f}%')
print(f'  Peak MAPE (>90% Actuals): {test_metrics["peak_mape"]:.3f}%')
print(f'  R^2 : {test_metrics["r2"]:.4f}')
print(f'  Samples used for Peak MAPE: {test_metrics["peak_samples"]} / {len(test_actuals)}')
print('='*60)

In [ ]:
results = pd.DataFrame({
    'datetime': time_test,
    'actual_demand_mw': test_actuals,
    'predicted_demand_mw': test_preds,
})

plt.figure(figsize=(15, 6))
plot_frame = results.tail(500).reset_index(drop=True)
plt.plot(plot_frame['datetime'], plot_frame['actual_demand_mw'], label='Actual', linewidth=1.5, alpha=0.8)
plt.plot(plot_frame['datetime'], plot_frame['predicted_demand_mw'], label='Predicted', linewidth=1.5, alpha=0.8)
plt.title('Gujarat Hourly Demand Forecast on Test Window (Last 500 hours)', fontsize=13)
plt.xlabel('Datetime', fontsize=11)
plt.ylabel('Demand MW', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

artifact_path = ARTIFACT_DIR / 'gujarat_lstm_regularized.pt'
scaler_path = SCALER_DIR / 'gujarat_lstm_regularized_scalers.pkl'

torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': X_train.shape[-1],
    'lookback': LOOKBACK,
    'feature_columns': feature_columns,
    'target_column': target_column,
    'metrics': {
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics,
    },
}, artifact_path)
joblib.dump({
    'x_scaler': x_scaler,
    'y_scaler': y_scaler,
}, scaler_path)

print(f'\nSaved model to: {artifact_path}')
print(f'Saved scalers to: {scaler_path}')

In [ ]:
# MAPE on lower 90% demand region (actuals below 90th percentile)
p90_threshold = np.percentile(test_actuals, 90)
lt90_mask = test_actuals < p90_threshold

peak_mape_lt90 = (
    np.mean(
        np.abs(
            (test_actuals[lt90_mask] - test_preds[lt90_mask]) /
            np.maximum(np.abs(test_actuals[lt90_mask]), 1e-6)
        )
    ) * 100
)

print(f'90th percentile threshold: {p90_threshold:.3f} MW')
print(f'MAPE for <90th percentile data: {peak_mape_lt90:.3f}%')
print(f'Samples used: {lt90_mask.sum()} / {len(test_actuals)}')

In [ ]:
from sklearn.model_selection import TimeSeriesSplit


def build_fold_dataset(frame, feature_cols, target_col, lookback, train_indices, eval_indices):
    fold_features = frame.iloc[np.r_[train_indices, eval_indices]].reset_index(drop=True)
    fold_values = fold_features[feature_cols].to_numpy(dtype=np.float32)
    fold_targets = fold_features[target_col].to_numpy(dtype=np.float32).reshape(-1, 1)
    fold_baseline = fold_features['demand_mw'].to_numpy(dtype=np.float32)
    fold_times = fold_features['datetime'].to_numpy()

    fold_x_scaler = StandardScaler()
    fold_y_scaler = StandardScaler()
    fold_x_scaler.fit(fold_values[:len(train_indices)])
    fold_y_scaler.fit(fold_targets[:len(train_indices)])

    fold_x_scaled = fold_x_scaler.transform(fold_values).astype(np.float32)
    fold_y_scaled = fold_y_scaler.transform(fold_targets).astype(np.float32).reshape(-1)
    X_fold, y_fold, t_fold, base_fold = build_sequences(
        fold_x_scaled,
        fold_y_scaled,
        fold_times,
        fold_baseline,
        LOOKBACK,
    )

    fold_target_positions = np.arange(LOOKBACK - 1, len(fold_features))
    train_mask_fold = fold_target_positions < len(train_indices)
    eval_mask_fold = fold_target_positions >= len(train_indices)

    X_train_fold = X_fold[train_mask_fold]
    y_train_fold = y_fold[train_mask_fold]
    X_eval_fold = X_fold[eval_mask_fold]
    y_eval_fold = y_fold[eval_mask_fold]
    base_eval_fold = base_fold[eval_mask_fold]
    time_eval_fold = t_fold[eval_mask_fold]

    train_ds_fold = TensorDataset(torch.from_numpy(X_train_fold), torch.from_numpy(y_train_fold))
    eval_ds_fold = TensorDataset(torch.from_numpy(X_eval_fold), torch.from_numpy(y_eval_fold))
    train_loader_fold = DataLoader(train_ds_fold, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    eval_loader_fold = DataLoader(eval_ds_fold, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    fold_model = RegularizedLSTM(input_dim=X_train_fold.shape[-1]).to(DEVICE)
    fold_criterion = nn.MSELoss()
    fold_optimizer = torch.optim.AdamW(fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    fold_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(fold_optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)

    best_fold_state = None
    best_fold_loss = float('inf')
    fold_patience = 0

    for epoch in range(1, 31):
        fold_model.train()
        for batch_x, batch_y in train_loader_fold:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            fold_optimizer.zero_grad(set_to_none=True)
            preds = fold_model(batch_x)
            loss = fold_criterion(preds, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fold_model.parameters(), GRAD_CLIP)
            fold_optimizer.step()

        fold_model.eval()
        eval_loss_total = 0.0
        eval_items = 0
        with torch.no_grad():
            for batch_x, batch_y in eval_loader_fold:
                batch_x = batch_x.to(DEVICE)
                batch_y = batch_y.to(DEVICE)
                preds = fold_model(batch_x)
                loss = fold_criterion(preds, batch_y)
                eval_loss_total += loss.item() * batch_x.size(0)
                eval_items += batch_x.size(0)
        fold_val_loss = eval_loss_total / max(eval_items, 1)
        fold_scheduler.step(fold_val_loss)

        if fold_val_loss < best_fold_loss - 1e-4:
            best_fold_loss = fold_val_loss
            best_fold_state = copy.deepcopy(fold_model.state_dict())
            fold_patience = 0
        else:
            fold_patience += 1
            if fold_patience >= 5:
                break

    if best_fold_state is not None:
        fold_model.load_state_dict(best_fold_state)

    fold_model.eval()
    fold_preds = []
    fold_actuals = []
    with torch.no_grad():
        for batch_x, batch_y in eval_loader_fold:
            batch_x = batch_x.to(DEVICE)
            fold_preds.append(fold_model(batch_x).cpu().numpy())
            fold_actuals.append(batch_y.numpy())

    fold_preds = np.concatenate(fold_preds)
    fold_actuals = np.concatenate(fold_actuals)
    fold_pred_actual = base_eval_fold + fold_y_scaler.inverse_transform(fold_preds.reshape(-1, 1)).reshape(-1)
    fold_true_actual = base_eval_fold + fold_y_scaler.inverse_transform(fold_actuals.reshape(-1, 1)).reshape(-1)

    fold_metrics = compute_metrics(fold_true_actual, fold_pred_actual)
    return {
        'rmse': fold_metrics['rmse'],
        'mae': fold_metrics['mae'],
        'mape': fold_metrics['mape'],
        'r2': fold_metrics['r2'],
        'peak_mape': fold_metrics['peak_mape'],
        'peak_samples': fold_metrics['peak_samples'],
        'start': time_eval_fold.min(),
        'end': time_eval_fold.max(),
    }


fold_results = []
tscv = TimeSeriesSplit(n_splits=3)
for fold_index, (train_idx, eval_idx) in enumerate(tscv.split(df), start=1):
    fold_info = build_fold_dataset(df, feature_columns, target_column, LOOKBACK, train_idx, eval_idx)
    fold_results.append(fold_info)
    print(
        f'Fold {fold_index}: '
        f'RMSE={fold_info["rmse"]:.3f}, '
        f'MAE={fold_info["mae"]:.3f}, '
        f'MAPE={fold_info["mape"]:.3f}%, '
        f'R^2={fold_info["r2"]:.4f}, '
        f'Peak MAPE={fold_info["peak_mape"]:.3f}%'
    )

fold_summary = pd.DataFrame(fold_results)
print('\nWalk-forward summary:')
print(fold_summary[['rmse', 'mae', 'mape', 'r2', 'peak_mape']].agg(['mean', 'std']).round(3))

## Walk-Forward Validation

This section evaluates generalization with chronological folds inside the **baseline horizon** (2020-01-01 to 2024-06-30).

The period **2024-07-01 to 2025-06-30** is reserved for a separate quarter-wise walk-forward benchmark plan and is intentionally excluded from baseline train/validation/test in this notebook.

## Notes

- The model predicts the next hour of demand using only past information and weather/calendar features.
- The architecture is intentionally large: stacked bidirectional LSTMs plus attention pooling and a deeper dense head.
- If you want, the next step is to turn this into a multi-step forecasting notebook or add a lightweight inference cell that loads the saved artifacts.